## 0. Installation and imports

In [1]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [3]:
import json
import random
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm

from PIL import Image
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score, roc_auc_score)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'timm    : {timm.__version__}')

## 1. Configuration

In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Trilha3-V2')
    IMGS_DIR = Path('/content/Imgs')
    import os
    if not os.path.exists('/content/Imgs'):
        print('Descompactando imagens para o disco rápido do Colab...')
        !unzip -q /content/drive/MyDrive/Trilha3-V2/Data/Imgs.zip -d /content/
        print('Descompactação concluída!')
else:
    BASE_DIR = Path('e:/Endo-ICTAI-2026')
    IMGS_DIR = Path('e:/Endo-ICTAI-2026/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'm2'
CKPT_DIR    = BASE_DIR / 'checkpoints'
FIGS_DIR    = BASE_DIR / 'figs' / 'training'

for d in [RESULTS_DIR, CKPT_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
N_FOLDS     = 5
N_CLASSES   = len(CORE_LABELS)

IMG_SIZE     = 224
BATCH_SIZE   = 128
MAX_EPOCHS   = 50
LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
ES_PATIENCE  = 10
LR_PATIENCE  = 4
LR_FACTOR    = 0.5
SEEDS        = [42, 43, 44, 45, 46]
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LAMBDA_VALUES = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7]

print(f'Device  : {DEVICE}')
print(f'λ sweep : {LAMBDA_VALUES}')

## 2. Dataset and transforms (AUG-02, same as NB02)

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(split: str) -> T.Compose:
    normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if split == 'train':
        return T.Compose([
            T.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
            T.RandomResizedCrop(size=IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.ToTensor(),
            normalize,
        ])
    else:
        return T.Compose([
            T.Resize((int(IMG_SIZE * 1.14), int(IMG_SIZE * 1.14))),
            T.CenterCrop(IMG_SIZE),
            T.ToTensor(),
            normalize,
        ])

class GastroscopyDataset(Dataset):
    def __init__(self, df, imgs_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.imgs_dir  = imgs_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, labels

def compute_pos_weights(df) -> torch.Tensor:
    pos = df[CORE_LABELS].sum(axis=0).values
    neg = len(df) - pos
    return torch.tensor(neg / (pos + 1e-6), dtype=torch.float32).to(DEVICE)

def compute_cooccurrence_matrix(df) -> torch.Tensor:
    """
    Computa matriz de co-ocorrência condicional a partir do conjunto de treino.
    C[i][j] = P(label_j=1 | label_i=1) — probabilidade de j dado i.
    Calculada por fold para evitar leakage.
    """
    labels = df[CORE_LABELS].values.astype(float)
    C = np.zeros((N_CLASSES, N_CLASSES))
    for i in range(N_CLASSES):
        mask_i = labels[:, i] == 1
        n_i    = mask_i.sum()
        if n_i == 0:
            continue
        for j in range(N_CLASSES):
            if i == j:
                continue
            C[i][j] = labels[mask_i, j].sum() / n_i
    return torch.tensor(C, dtype=torch.float32).to(DEVICE)

print('Dataset e funções auxiliares definidos.')

## 3. M2 Loss — Regularized co-occurrence

In [6]:
class M2CoocLoss(nn.Module):
    """
    L_M2 = L_BCE + λ * L_coo

    L_coo penaliza inconsistência entre probabilidades preditas e co-ocorrências
    clínicas observadas no treino:

        L_coo = Σ_{i≠j} C[i][j] * relu(p_i - p_j)^2

    Se C[i][j] é alto (quando i está presente, j tende a estar),
    o modelo é penalizado quando p_i > p_j — ou seja, prediz i mas não j.

    A matriz C é computada por fold a partir do treino — sem leakage.
    """
    def __init__(self, pos_weight: torch.Tensor, cooc_matrix: torch.Tensor, lam: float):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.C    = cooc_matrix
        self.lam  = lam

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_bce = self.bce(logits, targets)

        if self.lam == 0.0:
            return loss_bce

        probs = torch.sigmoid(logits)

        p_i   = probs.unsqueeze(2)
        p_j   = probs.unsqueeze(1)
        diff  = torch.relu(p_i - p_j)

        loss_coo = (self.C.unsqueeze(0) * diff.pow(2)).mean()

        return loss_bce + self.lam * loss_coo

print('M2CoocLoss definida.')

## 4. Model (Swin-Tiny) and optimizer

In [7]:
def build_swin_tiny(n_classes: int = N_CLASSES) -> nn.Module:
    return timm.create_model(
        'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
        pretrained=True,
        num_classes=n_classes
    )

def build_optimizer(model: nn.Module):
    head_params     = list(model.head.parameters())
    backbone_params = [p for n, p in model.named_parameters() if 'head' not in n]
    return torch.optim.AdamW(
        [
            {'params': backbone_params, 'lr': LR_BACKBONE},
            {'params': head_params,     'lr': LR_HEAD},
        ],
        weight_decay=WEIGHT_DECAY,
    )

print('Funções de modelo definidas.')

## 5. Metrics

In [8]:
def optimize_thresholds(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    thresholds = np.zeros(N_CLASSES)
    for i in range(N_CLASSES):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            if preds.sum() == 0:
                continue
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[i] = best_t
    return thresholds

def compute_metrics(probs: np.ndarray, labels: np.ndarray,
                    thresholds: np.ndarray = None) -> dict:
    if thresholds is None:
        thresholds = np.full(N_CLASSES, 0.5)
    preds = (probs >= thresholds).astype(int)

    f1_per   = f1_score(labels, preds, average=None, zero_division=0)
    pr_per   = precision_score(labels, preds, average=None, zero_division=0)
    rc_per   = recall_score(labels, preds, average=None, zero_division=0)

    auprc, rocauc = [], []
    for i in range(N_CLASSES):
        auprc.append(average_precision_score(labels[:, i], probs[:, i])
                     if labels[:, i].sum() > 0 else float('nan'))
        rocauc.append(roc_auc_score(labels[:, i], probs[:, i])
                      if 0 < labels[:, i].sum() < len(labels) else float('nan'))

    multi_mask = labels.sum(axis=1) >= 2
    f1_multi   = float(f1_score(labels[multi_mask], preds[multi_mask],
                                 average='macro', zero_division=0))\
                 if multi_mask.sum() > 0 else float('nan')

    result = {
        'f1_macro':      float(np.nanmean(f1_per)),
        'f1_micro':      float(f1_score(labels, preds, average='micro', zero_division=0)),
        'hamming':       float(np.mean(preds != labels)),
        'f1_multi':      f1_multi,
        'pr_auc_macro':  float(np.nanmean(auprc)),
        'roc_auc_macro': float(np.nanmean(rocauc)),
    }
    for i, label in enumerate(CORE_LABELS):
        result[f'f1_{label}']     = float(f1_per[i])
        result[f'prec_{label}']   = float(pr_per[i])
        result[f'rec_{label}']    = float(rc_per[i])
        result[f'auprc_{label}']  = float(auprc[i])
        result[f'rocauc_{label}'] = float(rocauc[i])
        result[f'thr_{label}']    = float(thresholds[i])
    return result

print('Funções de métricas definidas.')

## 6. Training loop

In [10]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = True

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        logits = model(imgs.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

def train_one_fold(fold: int, lam: float,
                   save_checkpoint: bool = True,
                   verbose: bool = True) -> dict:
    """
    Treina Swin-Tiny com M2 (lam>0) ou M0 (lam=0) em um fold.
    Co-occurrence matrix computada exclusivamente no treino do fold.
    """
    set_seed(SEEDS[fold])

    df_tr  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_train.csv')
    df_val = pd.read_csv(SPLITS_DIR / f'fold_{fold}_val.csv')
    df_te  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_test.csv')

    ds_tr  = GastroscopyDataset(df_tr,  IMGS_DIR, get_transforms('train'))
    ds_val = GastroscopyDataset(df_val, IMGS_DIR, get_transforms('val'))
    ds_te  = GastroscopyDataset(df_te,  IMGS_DIR, get_transforms('val'))

    dl_tr  = DataLoader(ds_tr,  batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
    dl_te  = DataLoader(ds_te,  batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)

    model     = build_swin_tiny().to(DEVICE)
    pos_w     = compute_pos_weights(df_tr)
    cooc_mat  = compute_cooccurrence_matrix(df_tr)
    criterion = M2CoocLoss(pos_w, cooc_mat, lam)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=LR_PATIENCE, factor=LR_FACTOR
    )

    best_val_f1  = -1.0
    best_weights = None
    es_counter   = 0
    history      = {'train_loss': [], 'val_f1': []}

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for imgs, labels in dl_tr:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(imgs)

        train_loss = running_loss / len(ds_tr)

        val_probs, val_labels = evaluate(model, dl_val)
        val_thr = optimize_thresholds(val_probs, val_labels)
        val_f1  = compute_metrics(val_probs, val_labels, val_thr)['f1_macro']

        scheduler.step(val_f1)
        history['train_loss'].append(train_loss)
        history['val_f1'].append(val_f1)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Epoch {epoch:>3} | loss={train_loss:.4f} | val_F1={val_f1:.4f}')

        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1  = val_f1
            best_weights = deepcopy(model.state_dict())
            es_counter   = 0
        else:
            es_counter += 1
            if es_counter >= ES_PATIENCE:
                if verbose:
                    print(f'  Early stopping na época {epoch}.')
                break

    model.load_state_dict(best_weights)
    if save_checkpoint:
        torch.save(best_weights, CKPT_DIR / f'swin_m2_l{int(lam*10):02d}_fold{fold}.pt')

    test_probs, test_labels = evaluate(model, dl_te)
    test_met = compute_metrics(test_probs, test_labels, val_thr)
    test_met.update({
        'fold': fold, 'lam': lam,
        'best_val_f1': best_val_f1,
        'epochs_trained': len(history['train_loss']),
        'val_thresholds': val_thr.tolist(),
        'history': history,
        'test_preds': (test_probs >= val_thr).astype(int).tolist(),
        'test_labels': test_labels.tolist(),
    })
    return test_met

print('Loop de treino definido.')

## 7. Execution — λ sweep

In [11]:
RUN_FAST_CHECK = False

if RUN_FAST_CHECK:
    MAX_EPOCHS   = 5
    FOLDS_TO_RUN = [0]
    print('MODO FAST CHECK: 1 fold, 5 épocas.')
else:
    FOLDS_TO_RUN = list(range(N_FOLDS))
    print(f'MODO COMPLETO: {N_FOLDS} folds × {len(LAMBDA_VALUES)} λ = '
          f'{N_FOLDS * len(LAMBDA_VALUES)} runs.')
    print(f'Tempo estimado (A100): ~{N_FOLDS * len(LAMBDA_VALUES) * 10 / 60:.0f}h')

In [12]:
all_results = {}

for lam in LAMBDA_VALUES:
    exp_id = 'm0_swin' if lam == 0.0 else f'm2_l{int(lam*10):02d}'

    print(f'\n{"="*60}')
    print(f'Experimento: {exp_id}  (λ={lam})')
    print(f'{"="*60}')

    fold_results = []
    t0 = time.time()

    for fold in FOLDS_TO_RUN:
        print(f'\n  Fold {fold}/{N_FOLDS-1}')
        met = train_one_fold(fold, lam, save_checkpoint=not RUN_FAST_CHECK)
        fold_results.append(met)
        print(f'  → Test Macro-F1={met["f1_macro"]:.4f}  '
              f'PÓLIPO={met["f1_PÓLIPO"]:.4f}  '
              f'MICRONOD={met["f1_MICRONODULARIDADE"]:.4f}')

    elapsed = time.time() - t0

    metric_keys = (['f1_macro', 'f1_micro', 'hamming', 'f1_multi',
                    'pr_auc_macro', 'roc_auc_macro'] +
                   [f'f1_{l}' for l in CORE_LABELS] +
                   [f'auprc_{l}' for l in CORE_LABELS])

    agg = {}
    for k in metric_keys:
        vals = [r[k] for r in fold_results
                if not np.isnan(r.get(k, float('nan')))]
        if vals:
            agg[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}

    all_results[exp_id] = {
        'config':       {'lam': lam},
        'fold_results': fold_results,
        'aggregated':   agg,
        'elapsed_sec':  round(elapsed, 1),
    }

    print(f'\n  AGREGADO: Macro-F1 = '
          f'{agg["f1_macro"]["mean"]:.4f} ± {agg["f1_macro"]["std"]:.4f}')
    print(f'  Tempo: {elapsed/60:.1f} min')

    with open(RESULTS_DIR / 'm2_results.json', 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print('  Salvo parcialmente.')

## 8. Statistical analysis — Bootstrap

In [13]:
def bootstrap_compare(preds_a, labels_a, preds_b, labels_b, B=2000, seed=42):
    """Bootstrap pareado (B=2000) real usando predições salvas."""
    rng = np.random.default_rng(seed)
    n   = len(labels_a)
    deltas = []
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        f1_a = f1_score(labels_a[idx], preds_a[idx], average='macro', zero_division=0)
        f1_b = f1_score(labels_b[idx], preds_b[idx], average='macro', zero_division=0)
        deltas.append(f1_b - f1_a)

    deltas  = np.array(deltas)
    delta_obs = deltas.mean()
    ci_low  = float(np.percentile(deltas, 2.5))
    ci_high = float(np.percentile(deltas, 97.5))
    p_value = float(np.mean(deltas <= 0))
    return {'delta_mean': float(delta_obs), 'ci_95_low': ci_low, 'ci_95_high': ci_high, 'p_value': p_value, 'significant': ci_low > 0}

def collect_preds_labels(exp_id):
    """Reconstrói preds e labels de teste concatenando os 5 folds."""
    all_p, all_l = [], []
    for r in all_results[exp_id]['fold_results']:
        all_p.extend(r['test_preds'])
        all_l.extend(r['test_labels'])
    return np.array(all_p), np.array(all_l)

preds_base, labels_base = collect_preds_labels('m0_swin')
base_f1 = f1_score(labels_base, preds_base, average='macro', zero_division=0)
print(f'M0 Swin-Tiny (baseline agrupado): {base_f1:.4f}')
print()
comparisons = {}
print(f'{"Experimento":<12} {"F1-macro":>10} {"Δ vs M0":>8} {"Bootstrap p":>12} {"Significativo":>14}')
print('-' * 65)
for exp_id, res in all_results.items():
    if exp_id == 'm0_swin': continue
    preds_exp, labels_exp = collect_preds_labels(exp_id)
    exp_f1 = f1_score(labels_exp, preds_exp, average='macro', zero_division=0)
    delta  = exp_f1 - base_f1

    boot = bootstrap_compare(preds_base, labels_base, preds_exp, labels_exp)
    p = boot['p_value']
    sig = '✓' if p < 0.05 else '—'
    lam = res['config']['lam']
    print(f'λ={lam:<5}       {exp_f1:>8.4f}  {delta:>+8.4f}  {p:>12.4f}  {sig:>14}')

    comparisons[exp_id] = {
        'lam': lam, 'f1_agrupado': float(exp_f1), 'delta_vs_m0': float(delta),
        'bootstrap_p': float(p), 'significant': bool(p < 0.05), 'ci_95_low': boot['ci_95_low'], 'ci_95_high': boot['ci_95_high']
    }

with open(RESULTS_DIR / 'bootstrap_comparisons.json', 'w', encoding='utf-8') as f:
    json.dump(comparisons, f, ensure_ascii=False, indent=2)
print('\nComparações salvas.')

## 9. Figure — Curve λ vs F1-macro

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lams   = [all_results[e]['config']['lam'] for e in all_results]
means  = [all_results[e]['aggregated']['f1_macro']['mean'] for e in all_results]
stds   = [all_results[e]['aggregated']['f1_macro']['std']  for e in all_results]

ax = axes[0]
ax.errorbar(lams, means, yerr=stds, marker='o', linewidth=2,
            capsize=5, color='#1565C0', markerfacecolor='white', markeredgewidth=2)

best_idx = int(np.argmax(means))
ax.scatter([lams[best_idx]], [means[best_idx]], color='#D32F2F', zorder=5,
           s=120, label=f'Melhor λ={lams[best_idx]} (F1={means[best_idx]:.3f})')

ax.axhline(0.537, color='gray', linestyle=':', linewidth=1.5,
           label='Trilha-3 M2 (ResNet50, λ=0.3): 0.537')
ax.axhline(means[0], color='orange', linestyle='--', linewidth=1.5,
           label=f'M0 Swin-Tiny (λ=0): {means[0]:.3f}')

ax.set_xlabel('λ (peso da regularização de co-ocorrência)', fontsize=11)
ax.set_ylabel('Macro-F1 (média ± std, 5 folds)', fontsize=11)
ax.set_title('Ablação de λ — M2 + Swin-Tiny', fontweight='bold')
ax.legend(fontsize=8)
ax.set_xticks(lams)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
heat_data = {}
for exp_id, res in all_results.items():
    lam_label = f'λ={res["config"]["lam"]}'
    heat_data[lam_label] = [
        res['aggregated'].get(f'f1_{l}', {}).get('mean', float('nan'))
        for l in CORE_LABELS
    ]

df_heat = pd.DataFrame(heat_data, index=CORE_LABELS).T
sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.2, vmax=0.85, linewidths=0.4, ax=ax,
            cbar_kws={'label': 'F1'})
ax.set_title('F1 por classe × λ', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')

plt.suptitle('M2 Co-ocorrência + Swin-Tiny — Ablação de λ', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'm2_lambda_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 10. Comparative table and final summary

In [15]:
REF_TRILHA3_M0 = {'f1_macro': 0.517, 'f1_ENANTEMA': 0.538, 'f1_PÓLIPO': 0.507,
                   'f1_ÚLCERA': 0.520, 'f1_EROSÃO': 0.670, 'f1_MICRONODULARIDADE': 0.351}
REF_TRILHA3_M2 = {'f1_macro': 0.537, 'f1_ENANTEMA': 0.470, 'f1_PÓLIPO': 0.624,
                   'f1_ÚLCERA': 0.520, 'f1_EROSÃO': 0.628, 'f1_MICRONODULARIDADE': 0.409}

rows = []
for label, ref in [('Trilha-3 M0 (ResNet50)', REF_TRILHA3_M0),
                    ('Trilha-3 M2 λ=0.3 (ResNet50)', REF_TRILHA3_M2)]:
    row = {'Modelo': label}
    for k in ['f1_macro', 'f1_ENANTEMA', 'f1_PÓLIPO', 'f1_ÚLCERA',
               'f1_EROSÃO', 'f1_MICRONODULARIDADE']:
        row[k.replace('f1_', '')] = f"{ref[k]:.3f}"
    rows.append(row)

for exp_id, res in all_results.items():
    agg = res['aggregated']
    lam = res['config']['lam']
    label = f"M0 Swin-Tiny" if lam == 0.0 else f"M2 Swin-Tiny λ={lam}"
    row = {'Modelo': label}
    for k in ['f1_macro', 'f1_ENANTEMA', 'f1_PÓLIPO', 'f1_ÚLCERA',
               'f1_EROSÃO', 'f1_MICRONODULARIDADE']:
        v = agg.get(k)
        row[k.replace('f1_', '')] = f"{v['mean']:.3f}±{v['std']:.3f}" if v else '-'
    rows.append(row)

df_table = pd.DataFrame(rows).set_index('Modelo')
print('=== TABELA COMPARATIVA COMPLETA ===')
print(df_table.to_string())
df_table.to_csv(RESULTS_DIR / 'tabela_m2.csv')

print('\n=== RESUMO FINAL ===')
best_exp = max(
    [(e, r['aggregated']['f1_macro']['mean']) for e, r in all_results.items()],
    key=lambda x: x[1]
)
print(f'Melhor configuração : {best_exp[0]} (F1={best_exp[1]:.4f})')
print(f'M0 Swin-Tiny        : {all_results["m0_swin"]["aggregated"]["f1_macro"]["mean"]:.4f}')
print(f'Trilha-3 M2 (ref)   : 0.537')
m0 = all_results['m0_swin']['aggregated']['f1_macro']['mean']
print(f'Δ melhor M2 vs M0   : {best_exp[1] - m0:+.4f}')
print(f'Δ melhor M2 vs T3-M2: {best_exp[1] - 0.537:+.4f}')
print()
print('Este backbone (Swin-Tiny) + M2 otimizado será usado no NB04 (WRS + Focal Loss).')

In [16]:
import time
print('Treinamento finalizado. A sessão será encerrada em 5 minutos para economizar créditos do Colab.')
time.sleep(300)
if IN_COLAB:
    from google.colab import runtime
    runtime.unassign()